In [1]:
from amstools import QAtoms

In [4]:
from ase.build import fcc100, bulk

In [ ]:
# create cubuc Al-fcc and build 2x2x2 supercell
qat=QAtoms.bulk("Al", cubic=True)*(2,2,2)

In [ ]:
# all atoms are selected by default
qat

In [6]:
# Alternatively, just wrap any ase.Atoms with QAtoms
qat=QAtoms(bulk("Al", cubic=True)*(2,2,2))
qat

QAtoms(symbols='Al32', pbc=True, cell=[8.1, 8.1, 8.1], 32 selected atoms)

In [7]:
# one can visualize atoms 
# qat.view()

2026-01-14 00:03:19.477 python[78859:23930506] +[IMKClient subclass]: chose IMKClient_Modern
2026-01-14 00:03:19.477 python[78859:23930506] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [6]:
# randomly select 1 atom
qat.sample(n=1,random_state=42)

QAtoms(symbols='Al32', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 1 selected atoms)

# IMPORTANT NOTES!

1. Any operation on QAtoms **ALWAYS** return a modified **COPY**. Original object is not changed
2. As a sequence, you can chain operations, i.e. `qatoms.method1().method2()`, etc. 

# Substitution

In [21]:
# randomly select and substitute with 'Li'
# selection moves to the new atom(s)
r = qat.sample(n=1,random_state=1).set(element="Li")
r

QAtoms(symbols='Al27LiAl4', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 1 selected atoms)

In [22]:
# select nearest atom to 'Li', replace it with Ni, then select all atoms within 3A cutoff, replace all elements with Cu
r.select_nn().set(element='Ni').select_nearby(cutoff=3).set(element='Cu')#.view()

QAtoms(symbols='Al3CuAl6Cu2Al2CuAl2CuAlCuAlCuAl2NiCu3AlCu2Al', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 12 selected atoms)

In [9]:
# Starting from original structure (qat), select 1 atom randomly, set it to Li, 
# find two nearest neighbours to Li and delete them
qat.sample(n=1,random_state=42).set(element="Li").select_nn(n=2).delete()#.view()

QAtoms(symbols='Al28LiAl', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 0 selected atoms)

In [10]:
# qat.sample(n=3,random_state=42).flat().set(element='Ni').select_nn().set(element='Li').view()

# Add interstitial

In [11]:
# select one atom randomly
r=qat.sample(n=1,random_state=42)
r

QAtoms(symbols='Al32', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 1 selected atoms)

In [12]:
# Add H interstitial randomly within 2.5 A distance from selected atom, but not closer than 1 A
# Do maximum 100 attempts. Move selection to newly inserted H atoms.
# Those are DEFAULTS values 
r.insert_interstitial(element='H', cutoff=2.5, min_dist=1., max_attempts=100, select_new=True)

QAtoms(symbols='Al32H', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 1 selected atoms)

In [13]:
# Set selected atom to Li. 
# Add H interstitial as before, but leave the selection on Li
# Insert second H interstitial near Li
Al_Li_H2=r.set(element='Li').insert_interstitial(element='H', select_new=False).insert_interstitial(element='H')#.view()
Al_Li_H2

QAtoms(symbols='Al29LiAl2H2', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 1 selected atoms)

In [14]:
# select all H and replace them with He
Al_Li_H2.select(element='H',from_all=True).set(element='He')#.view()

QAtoms(symbols='Al29LiAl2He2', pbc=True, cell=[8.1, 8.1, 8.1], _selected_mask=..., 2 selected atoms)

# Manipulations with tags

In [15]:
# create surface FCC (100) with ase.build.fcc100
surf=fcc100('Al', (2,2,5),periodic=True,vacuum=10)
surf

Atoms(symbols='Al20', pbc=True, cell=[5.727564927611035, 5.727564927611035, 28.1], tags=...)

In [16]:
# wrap to QAtoms
q=QAtoms(surf)
q

QAtoms(symbols='Al20', pbc=True, cell=[5.727564927611035, 5.727564927611035, 28.1], tags=..., 20 selected atoms)

In [17]:
# first four surface atoms have tag=5
q.get_tags()[:4]

array([5, 5, 5, 5])

In [18]:
# select surface atoms
q_surf=q.select(tag=5)
q_surf

QAtoms(symbols='Al20', pbc=True, cell=[5.727564927611035, 5.727564927611035, 28.1], _selected_mask=..., tags=..., 4 selected atoms)

In [19]:
# build supercell 2x2x1
# Note! selection will also be multiplied and extended
q_surf_supercell = q_surf*(2,2,1)
q_surf_supercell

QAtoms(symbols='Al80', pbc=True, cell=[11.45512985522207, 11.45512985522207, 28.1], _selected_mask=..., tags=..., 16 selected atoms)

In [23]:
# select one (surface) atoms, set to Ni, select nearby atoms within to 3 A distance, 
# again select only those which have tag=5 (surface) and set them to Li
# Result: Ni atoms , surrounded by Ni atoms, but only on the first surface layer
q_surf_supercell.sample(n=1).set(element='Ni').select_nearby(cutoff=3).select(tag=5).set(element='Li')#.view()

QAtoms(symbols='Al75Li4Ni', pbc=True, cell=[11.45512985522207, 11.45512985522207, 28.1], _selected_mask=..., tags=..., 4 selected atoms)

# TODO: QAtomsCollection

In [8]:
from amstools.qatoms import QAtomsCollection